In [1]:
import pandas as pd
import re
from pathlib import Path

team_data_dir = Path("data/team-data")

# Already has all 16 target columns
akash_df = pd.read_json(team_data_dir / "akash.jsonl", lines=True)

# Format A: raw pod_describe / pod_logs / pod_logs_previous
mrunali_df = pd.read_json(team_data_dir / "mrunali.jsonl", lines=True)

# Format B: pre-parsed fields with evidence_text containing embedded describe
chels_df = pd.read_json(team_data_dir / "chels.jsonl", lines=True)
devesh_df = pd.read_json(team_data_dir / "devesh.jsonl", lines=True)
najel_df = pd.read_json(team_data_dir / "najel.jsonl", lines=True)

print(f"akash: {len(akash_df)}")
print(f"mrunali: {len(mrunali_df)}")
print(f"chels: {len(chels_df)}, devesh: {len(devesh_df)}, najel: {len(najel_df)}")
print(f"Total: {len(akash_df) + len(mrunali_df) + len(chels_df) + len(devesh_df) + len(najel_df)}")

akash: 1500
mrunali: 1000
chels: 1500, devesh: 1500, najel: 1500
Total: 7000


In [2]:
def normalize_text(x):
    if pd.isna(x):
        return ""
    x = str(x).strip()
    if x.lower() in {"nan", "none"}:
        return ""
    x = x.replace("\r\n", "\n").replace("\r", "\n").replace("\t", "    ")
    lines = [line.rstrip() for line in x.split("\n")]
    return "\n".join(lines).strip()

def extract_first(pattern, text, flags=re.IGNORECASE | re.MULTILINE):
    if not text:
        return None
    m = re.search(pattern, text, flags)
    return m.group(1).strip() if m else None

def extract_block(text, block_name):
    if not text:
        return None
    lines = text.split("\n")
    block = []
    in_block = False
    for line in lines:
        if not in_block:
            if re.match(rf"^{re.escape(block_name)}:\s*$", line.strip()):
                in_block = True
            continue
        if line and not line.startswith(" ") and re.match(r"^[A-Za-z0-9_.\-/ ]+:\s*", line):
            break
        block.append(line)
    out = "\n".join(block).strip()
    return out if out else None

def parse_events_block(events_text):
    if not events_text:
        return {"event_reason": None, "event_message": None}
    reasons, messages = [], []
    lines = [line for line in events_text.split("\n") if line.strip()]
    for line in lines:
        if "Reason" in line and "Message" in line:
            continue
        if line.strip().startswith("----"):
            continue
        parts = re.split(r"\s{2,}", line.strip())
        if len(parts) >= 5:
            reasons.append(parts[1].strip())
            messages.append(parts[-1].strip())
    return {
        "event_reason": reasons[0] if reasons else None,
        "event_message": messages[0] if messages else None,
    }

def parse_describe(text):
    containers_block = extract_block(text, "Containers")
    volumes_block = extract_block(text, "Volumes")
    events_block = extract_block(text, "Events")
    events = parse_events_block(events_block)
    return {
        "pod_name": extract_first(r"^Name:\s*(.+)$", text),
        "namespace": extract_first(r"^Namespace:\s*(.+)$", text),
        "service_account_name": extract_first(r"^Service Account:\s*(.+)$", text),
        "node": extract_first(r"^Node:\s*(.+)$", text),
        "pod_status": extract_first(r"^Status:\s*(.+)$", text),
        "image": extract_first(r"^\s*Image:\s*(.+)$", containers_block or text),
        "container_state": extract_first(r"^\s*State:\s*(.+)$", containers_block or text),
        "last_state": extract_first(r"^\s*Last State:\s*(.+)$", containers_block or text),
        "ready": extract_first(r"^\s*Ready:\s*(.+)$", containers_block or text),
        "restart_count": extract_first(r"^\s*Restart Count:\s*(.+)$", containers_block or text),
        "node_selectors": extract_first(r"^Node-Selectors:\s*(.+)$", text),
        "claim_name": extract_first(r"^\s*ClaimName:\s*(.+)$", volumes_block or ""),
        "event_reason": events["event_reason"],
        "event_message": events["event_message"],
    }

def build_evidence_text(row):
    parts = []
    if row.get("pod_describe"):
        parts.append("POD DESCRIBE:\n" + row["pod_describe"])
    if row.get("pod_logs"):
        parts.append("POD LOGS:\n" + row["pod_logs"])
    if row.get("pod_logs_previous"):
        parts.append("POD LOGS PREVIOUS:\n" + row["pod_logs_previous"])
    return "\n\n".join(parts).strip() if parts else None

def extract_describe_from_evidence(evidence_text):
    if not evidence_text:
        return ""
    m = re.search(r"=== kubectl describe pod ===\n(.*?)(?:\n=== |\Z)", evidence_text, re.DOTALL)
    return m.group(1).strip() if m else ""

In [3]:
# --- Process Format A (mrunali): parse pod_describe into structured fields ---
raw_cols = ["scenario_id", "pod_describe", "pod_logs", "pod_logs_previous"]
format_a = mrunali_df[raw_cols].copy()

for col in ["pod_describe", "pod_logs", "pod_logs_previous"]:
    format_a[col] = format_a[col].apply(normalize_text)

parsed_a = format_a["pod_describe"].apply(parse_describe).apply(pd.Series)
format_a = pd.concat([format_a, parsed_a], axis=1)
format_a["evidence_text"] = format_a.apply(build_evidence_text, axis=1)

print(f"Format A (mrunali): {len(format_a)} rows")
format_a[["scenario_id", "pod_name", "namespace", "pod_status", "event_reason"]].head()

Format A (mrunali): 1000 rows


,scenario_id,pod_name,namespace,pod_status,event_reason
0,dns_resolution_failure,dns-fail-demo-wr6dw,mk-gsw3vog7,Failed,Scheduled
1,dns_resolution_failure,dns-fail-demo-2zct8,mk-8brb56t8,Failed,Scheduled
2,dns_resolution_failure,dns-fail-demo-7rmlh,mk-d9y1y2w1,Failed,Scheduled
3,dns_resolution_failure,dns-fail-demo-sb2sr,mk-t0l3y2x5,Failed,Scheduled
4,dns_resolution_failure,dns-fail-demo-s8rgx,mk-805hbsm4,Failed,Scheduled


In [4]:
# --- Process Format B (chels, devesh, najel): extract describe from evidence_text, parse missing fields ---
format_b = pd.concat([chels_df, devesh_df, najel_df], ignore_index=True)
format_b["evidence_text"] = format_b["evidence_text"].apply(normalize_text)

# Extract pod_describe, pod_logs, pod_logs_previous from evidence_text
def extract_section(evidence_text, section_name):
    if not evidence_text:
        return ""
    m = re.search(rf"=== {re.escape(section_name)} ===\n(.*?)(?:\n=== |\Z)", evidence_text, re.DOTALL)
    return m.group(1).strip() if m else ""

format_b["pod_describe"] = format_b["evidence_text"].apply(lambda x: extract_section(x, "kubectl describe pod"))
format_b["pod_logs"] = format_b["evidence_text"].apply(lambda x: extract_section(x, "container logs"))
format_b["pod_logs_previous"] = format_b["evidence_text"].apply(lambda x: extract_section(x, "container logs (previous)"))

# Parse the kubectl describe section for structured fields
parsed_b = format_b["pod_describe"].apply(parse_describe).apply(pd.Series)

for col in parsed_b.columns:
    if col not in format_b.columns:
        format_b[col] = parsed_b[col]
    else:
        format_b[col] = format_b[col].fillna(parsed_b[col])

print(f"Format B (chels + devesh + najel): {len(format_b)} rows")
format_b[["scenario_id", "pod_name", "namespace", "pod_status", "event_reason"]].head()

Format B (chels + devesh + najel): 4500 rows


,scenario_id,pod_name,namespace,pod_status,event_reason
0,createcontainerconfigerror_missing_secret,service-w-bm4b-0,team-b-stg-xqqv2a,Pending,Scheduled
1,createcontainerconfigerror_missing_secret,service-w-cmbj-69b87b95bd-4vpb4,orders-prod-sk53ni,Pending,Scheduled
2,createcontainerconfigerror_missing_secret,backend-w-ks1x-0,team-b-stg-qkb6mw,Pending,Scheduled
3,createcontainerconfigerror_missing_secret,api-w-s3zf-7b6946fdc-vw8gw,team-a-dev-0qpcfa,Pending,Scheduled
4,createcontainerconfigerror_missing_secret,service-w-01zx-0,orders-prod-gyn4fj,Pending,Scheduled


In [5]:
# --- Combine into final DataFrame with target columns ---
FINAL_COLS = [
    "scenario_id",
    "namespace",
    "pod_name",
    "service_account_name",
    "node",
    "pod_status",
    "image",
    "container_state",
    "last_state",
    "ready",
    "restart_count",
    "node_selectors",
    "claim_name",
    "event_reason",
    "event_message",
    "pod_describe",
    "pod_logs",
    "pod_logs_previous",
    "evidence_text",
]

combined_df = pd.concat([
    akash_df[FINAL_COLS],
    format_a[FINAL_COLS],
    format_b[FINAL_COLS],
], ignore_index=True)
print(f"Combined: {len(combined_df)} rows")
combined_df.info()

Combined: 7000 rows
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7000 entries, 0 to 6999
Data columns (total 19 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   scenario_id           7000 non-null   object
 1   namespace             7000 non-null   object
 2   pod_name              7000 non-null   object
 3   service_account_name  7000 non-null   object
 4   node                  7000 non-null   object
 5   pod_status            7000 non-null   object
 6   image                 7000 non-null   object
 7   container_state       4488 non-null   object
 8   last_state            1252 non-null   object
 9   ready                 4488 non-null   object
 10  restart_count         4488 non-null   object
 11  node_selectors        7000 non-null   object
 12  claim_name            512 non-null    object
 13  event_reason          7000 non-null   object
 14  event_message         7000 non-null   object
 15  pod_describe      

In [6]:
combined_df.head(10)

,scenario_id,namespace,pod_name,service_account_name,node,pod_status,image,container_state,last_state,ready,restart_count,node_selectors,claim_name,event_reason,event_message,pod_describe,pod_logs,pod_logs_previous,evidence_text
0,pvc_not_found_mountfail,data-pipeline,busybox-pn3zirfh-86bbb7957c-7754l,default,<none>,Pending,busybox:1.36,None,None,None,NaN,<none>,missing-pvc-pn3zirfh,FailedScheduling,0/1 nodes are available: persistentvolumeclaim...,Name: busybox-pn3zirfh-86bbb7957c-...,,,POD DESCRIBE:\nName: busybox-pn3zi...
1,rbac_forbidden,monitoring,kube-state-metrics-hnw5520p-7fbb7866df-8rsgh,app-sa-hnw5520p,kind-control-plane/172.18.0.5,Running,registry.k8s.io/kube-state-metrics/kube-state-...,Running,None,True,0.0,<none>,None,Scheduled,Successfully assigned monitoring/kube-state-me...,Name: kube-state-metrics-hnw5520p-...,I0331 06:37:30.545502 1 wrapper.go:120] ...,Error from server (BadRequest): previous termi...,POD DESCRIBE:\nName: kube-state-me...
2,rbac_forbidden,qa-app,ingress-nginx-4g3nkfre-656c6b79b6-vbbbn,app-sa-4g3nkfre,kind-control-plane/172.18.0.5,Running,registry.k8s.io/ingress-nginx/controller:v1.13.3,Waiting,Terminated,False,1.0,<none>,None,Scheduled,Successfully assigned qa-app/ingress-nginx-4g3...,Name: ingress-nginx-4g3nkfre-656c6...,----------------------------------------------...,----------------------------------------------...,POD DESCRIBE:\nName: ingress-nginx...
3,rbac_forbidden,prod-app,ingress-nginx-n5oblcdc-78d4488dfd-gnt5g,app-sa-n5oblcdc,kind-control-plane/172.18.0.5,Running,registry.k8s.io/ingress-nginx/controller:v1.13.3,Waiting,Terminated,False,1.0,<none>,None,Scheduled,Successfully assigned prod-app/ingress-nginx-n...,Name: ingress-nginx-n5oblcdc-78d44...,----------------------------------------------...,----------------------------------------------...,POD DESCRIBE:\nName: ingress-nginx...
4,rbac_forbidden,deployment,kube-state-metrics-yrnzw8kt-84b9496c49-qtsw4,app-sa-yrnzw8kt,kind-control-plane/172.18.0.5,Running,registry.k8s.io/kube-state-metrics/kube-state-...,Running,None,True,0.0,<none>,None,Scheduled,Successfully assigned deployment/kube-state-me...,Name: kube-state-metrics-yrnzw8kt-...,I0331 06:37:30.542740 1 wrapper.go:120] ...,Error from server (BadRequest): previous termi...,POD DESCRIBE:\nName: kube-state-me...
5,pvc_not_found_mountfail,app,redis-wuwdazqz-6cbddc486f-dt8vw,default,<none>,Pending,redis:7.2,None,None,None,NaN,<none>,missing-pvc-wuwdazqz,FailedScheduling,0/1 nodes are available: persistentvolumeclaim...,Name: redis-wuwdazqz-6cbddc486f-dt...,,,POD DESCRIBE:\nName: redis-wuwdazq...
6,pvc_not_found_mountfail,monitoring,nginx-0kbrd6uu-65c96d8888-5qkdp,default,<none>,Pending,nginx:1.27,None,None,None,NaN,<none>,missing-pvc-0kbrd6uu,FailedScheduling,0/1 nodes are available: persistentvolumeclaim...,Name: nginx-0kbrd6uu-65c96d8888-5q...,,,POD DESCRIBE:\nName: nginx-0kbrd6u...
7,pvc_not_found_mountfail,app-deployment,nginx-6ggb2my5-9b59965bd-rbdvh,default,<none>,Pending,nginx:1.27,None,None,None,NaN,<none>,missing-pvc-6ggb2my5,FailedScheduling,0/1 nodes are available: persistentvolumeclaim...,Name: nginx-6ggb2my5-9b59965bd-rbd...,,,POD DESCRIBE:\nName: nginx-6ggb2my...
8,pvc_not_found_mountfail,prod-app,nginx-99p30cd9-7cf76c86d4-qjkl7,default,<none>,Pending,nginx:1.27,None,None,None,NaN,<none>,missing-pvc-99p30cd9,FailedScheduling,0/1 nodes are available: persistentvolumeclaim...,Name: nginx-99p30cd9-7cf76c86d4-qj...,,,POD DESCRIBE:\nName: nginx-99p30cd...
9,rbac_forbidden,qa-app,kube-state-metrics-a7leoiiu-545cbf487c-jp7hq,app-sa-a7leoiiu,kind-control-plane/172.18.0.5,Running,registry.k8s.io/kube-state-metrics/kube-state-...,Running,None,True,0.0,<none>,None,Scheduled,Successfully assigned qa-app/kube-state-metric...,Name: kube-state-metrics-a7leoiiu-...,I0331 06:37:46.406699 1 wrapper.go:120] ...,Error from server (BadRequest): previous termi...,POD DESCRIBE:\nName: kube-state-me...


In [7]:
output_path = "data/02-raw/k8s_combined_incidents.jsonl"
combined_df.to_json(output_path, orient="records", lines=True)
print(f"Saved {len(combined_df)} rows to {output_path}")

Saved 7000 rows to data/02-raw/k8s_combined_incidents.jsonl
